
# Calibration run: 모든 arm 을 같은 각도로 두고 측정

positioner 들을 같은 명령각(예: 초기 위치 (0, 0) = 기하각 (30°, 150°))이나 회전 중심에서 같은 offset 으로 보낸 뒤 찍은 이미지로, 각 positioner 가 실제로 어디에 있는지와
목표까지의 회전량을 구한다. 목표 파일은 각도로 만들며(`lab.make_target`), `object.info` 자리에 그대로 쓸 수 있다
(`load_configuration` 이 각도도 읽는다). 나머지 흐름은 `Metrology_Debug.ipynb` 와 같다.


## 촬영 (take_exposure.py)
노출은 `kspec_metrology/notebooks/take_exposure.py` 로 직접 찍는다 (노출시간 [ms]; 이름 끝에 `_0` 이 자동으로 붙는다; `--dry-run` 은 이름만 확인).
trial 마다 세 장(fiber 1 + fiducial 2)을 한 번에:
```bash
python kspec_metrology/notebooks/take_exposure.py --fiber 400 --fid 2 10 --out /data/calib_t1     # trial 1
python kspec_metrology/notebooks/take_exposure.py --fiber 400 --fid 2 10 --out /data/calib_t2     # positioner 를 옮긴 뒤
```
끝에 찍히는 `FIBER_IMAGE =` / `FID_IMAGES =` 줄을 2절(trial 1) 또는 3절(반복) 에 붙여 넣는다.


In [ ]:

import os, sys, json, glob
PKG_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))   # 이 노트북이 kspec_metrology/notebooks/ 에 있을 때 저장소 루트
OUT_DIR  = os.path.expanduser('~/metrology_out')                   # <- 결과를 쓸 폴더

# ---- 설정: 바꾸면 커널을 재시작한다 (표와 제외 목록은 import 때 읽힌다) ----
FIBER_TABLE   = None            # None = 패키지 기본 표 (config73). 다른 표면 경로
# 빈 리스트 [] 는 패키지 기본값(analysis/fiber_config.py, 설치표 2026-10-12 기준) 을 그대로 쓴다. 적으면 그것이 우선.
BROKEN_FIBERS = []              # 빛이 안 나는 positioner 홀 ID. 기본: 분광기에 fiber 가 안 연결된 33 홀, 끊어진 G4 포함 (= cfg.FIBER_EXCLUDE)
FIXED_FIBERS  = []              # 켜져 있지만 움직이지 않는 positioner 홀 ID. 기본: ['G7', 'H8', 'J6'] (sky fiber 가 꽂힌 고정 positioner)
SKY_FIBERS    = []              # 표에서 -9 인 홀의 sky fiber. 기본: ['B7', 'H1', 'N7']
ANGLE_SIGN    = (-1, -1)        # 명령각 = 부호 x 기하각 - 영점. 본 장비 positioner 는 기하각(반시계 alpha, elbow-down beta)의 반대로 돈다 (2026-10-27 확인)
ANGLE_ZERO    = (-30, -150)     # positioner 명령각 영점: 기하각 (30, 150) = 명령 (0, 0) (부호 -1 이라 영점은 -30, -150). 출력·각도 입력 모두 명령각
MEASURED_POSITIONERS = True     # positioner 회전 중심을 arm scan 실측값(Measured_Geometry.txt)으로 쓸지. 핀홀 실측과는 별개 스위치 (2026-10-27 부터 기본 True)

if FIBER_TABLE:
    os.environ['KSPEC_FIBER_TABLE'] = FIBER_TABLE
os.environ['KSPEC_FIBER_EXCLUDE'] = ','.join(BROKEN_FIBERS)
os.environ['KSPEC_FIBER_FIXED']   = ','.join(FIXED_FIBERS)
os.environ['KSPEC_SKY_FIBERS']    = ','.join(SKY_FIBERS)
os.environ['KSPEC_ANGLE_SIGN']    = f'{ANGLE_SIGN[0]},{ANGLE_SIGN[1]}'
os.environ['KSPEC_ANGLE_ZERO']    = f'{ANGLE_ZERO[0]},{ANGLE_ZERO[1]}'
os.environ['KSPEC_USE_MEASURED']  = '1'                        # fiducial 핀홀 실측 위치 + arm 길이 측정표 (거의 항상 켬)
os.environ['KSPEC_USE_MEASURED_POSITIONERS'] = '1' if MEASURED_POSITIONERS else '0'
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, PKG_DIR)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 80
from kspec_metrology import lab
from kspec_metrology.analysis import fiber_config as cfg
print(f'fiber table {cfg.FIBER_TABLE_PATH}: {len(cfg.fiber_ids())} positioners')
print('broken', cfg.FIBER_EXCLUDE, '| fixed', cfg.FIBER_FIXED, '| sky', cfg.SKY_FIBERS, '| angle sign', cfg.ANGLE_SIGN, '| angle zero', cfg.ANGLE_ZERO)


In [ ]:

MAGNIFICATION  = None           # camera -> focal 배율 초기값. None = 패키지 기본 (2026-01 commissioning fit -8.86, 설치된 실카메라). 10/17 시험 셋업 영상은 8.40
MODEL          = 'full+affine'  # 본 장비. fiducial 30개 기준 leave-one-out 이 가장 좋았다 (README 표). 실험실 광학계는 'radial+affine'
NWINDOW        = 40
NITER_RECENTER = 1
THRESHOLD      = 1e3            # 시작 문턱 [ADU, 배경 포함]; 검출 개수에 맞춰 자동으로 올리고 내린다
MIN_DIST       = 80             # 이보다 가까운 검출은 하나로 합침 [px]
FINDER_OPTS    = {'minarea': 9, 'deblend_cont': 1.0}
N_EXTRA        = 0              # FIXED/SKY 외의 여분 밝은 점 (반사광 등)
SAT_LEVEL      = 65000.
TILE           = 'tile01'



## 1. 목표 파일
셋 중 하나. 배열은 표의 positioner 순서로 73개가 모두 채워진다 (끊어진/고정된 것도 포함; 그것들은 결과에서 NaN).


In [ ]:

TARGET_FILE = os.path.join(OUT_DIR, 'calibration_target.info')
lab.make_target(TARGET_FILE, angles=(0., 0.))            # 모두 같은 명령각 (alpha, beta) [deg]; (0, 0) = 기하각 (30, 150) = 초기 위치
# lab.make_target(TARGET_FILE, offset=(8., 0.))         # 모두 회전 중심에서 같은 (dx, dy) [mm]
print(open(TARGET_FILE).read()[:300], '...')



## 2. 측정 (trial 1)
이미지는 아무 이름이나 된다 (`notebooks/take_exposure.py` 로 찍어도 됨; `...0.fits` 로 끝나야 한다). `lab.measure` 가 `OUT_DIR/data` 에 패키지 이름 규칙으로 링크해 분석한다.
`itrial=1` 로 시작하고, 같은 이미지로 다시 계산만 할 때도 `itrial=1` 을 그대로 둔다.


In [ ]:

FIBER_IMAGE = '/path/to/fiber.fits'
FID_IMAGES  = []                                 # fiducial 전용 노출 0-2 장
TILE        = 'calib'
res = lab.measure(TARGET_FILE, FIBER_IMAGE, FID_IMAGES, out_dir=OUT_DIR, tile=TILE, itrial=1,
                  magnification=MAGNIFICATION, model=MODEL, nwindow=NWINDOW, threshold=THRESHOLD,
                  niter_recenter=NITER_RECENTER, sat_level=SAT_LEVEL, n_extra=N_EXTRA,
                  min_dist=MIN_DIST, finder_opts=FINDER_OPTS)
tab = lab.offsets_table(res); err = np.hypot(tab.dx_um, tab.dy_um)
print(f'fiber - target: median {np.nanmedian(err):.1f} um, max {np.nanmax(err):.1f} um')
lab.show_offsets(res['offsets']);
tab.round(3)



## 2b. 고정 positioner / sky fiber 자리 적기 (첫 trial 뒤 한 번)
sky fiber 가 꽂힌 고정 positioner(`cfg.FIBER_FIXED`, 3개)와 sky 홀의 fiber(`cfg.SKY_FIBERS`, 3개)는 움직이지 않으므로, 첫 측정에서 그 자리를 재어
`Fixed_Spots.txt` 로 적어 두면 이후 trial 에서 그 자리 1.5 mm 안의 검출은 fiber 후보에서 바로 빠진다 (살아 있는 fiber 가 목표에서
몇 mm 벗어나 있어도 고정 점과 헷갈리지 않는다). 위 2절 결과에서 `fiber - target` 경고(3 mm 초과)가 고정 점 오인으로 보이면 먼저 이것을 만든다.
`status == 'not found'` 가 있으면 그 홀의 점이 이 이미지에 안 보인 것이다 (threshold 를 낮추거나 목록을 확인).
파일은 패키지 안 `analysis/Fixed_Spots.txt` 가 기본이고 (`cfg.FIXED_SPOTS_PATH`), 다른 곳에 두면 `KSPEC_FIXED_SPOTS` 환경변수로 가리킨다.
적은 뒤 2절을 다시 실행하면 로그에 `Fixed spots: 6 detection(s) removed` 가 나온다.


In [ ]:

FIXED_SPOTS_OUT = cfg.FIXED_SPOTS_PATH           # 패키지 안 analysis/Fixed_Spots.txt (git 에 올라가지 않게 하려면 다른 경로 + KSPEC_FIXED_SPOTS)
F = lab.find_fixed_spots(res, out_path=FIXED_SPOTS_OUT)
F.round(3)



## 3. 반복 (positioner 를 옮긴 뒤)
`res['json']` 의 각도로 positioner 를 옮기고 → **fiber 노출과 fiducial 노출을 모두 다시 찍고** → 아래 셀에 새 이미지 경로들을 넣어 실행한다.
trial 마다 전 과정(fiducial 검출 → 변환 fit → fiber 검출 → 각도)을 처음부터 다시 하므로 fiducial 노출도 새로 준다
(`take_exposure.py --fiber 400 --fid 2 10 --out /data/calib_t2` 로 세 장을 한 번에). `FID_IMAGES_NEXT = []` 로 두면 fiducial 을 fiber 이미지에서 찾는다 (한 노출에 같이 찍었을 때만).
`itrial=None` 이면 `OUT_DIR` 에 있는 json 의 다음 번호가 되고 **누적 각도 = 직전 trial json + 이번 회전량** 으로 쌓인다.
같은 배치면 `OUT_DIR`/`TILE` 을 바꾸지 말 것. `lab.history` 가 trial 별 오차를 보여 준다.


In [ ]:

FIBER_IMAGE_NEXT = '/data/calib_t2_fib_400ms_0.fits'                                   # <- positioner 를 옮긴 뒤 찍은 fiber 노출
FID_IMAGES_NEXT  = ['/data/calib_t2_fid1_2ms_0.fits', '/data/calib_t2_fid2_10ms_0.fits']   # <- 같이 새로 찍은 fiducial 노출 (0-2 장)
res = lab.measure(TARGET_FILE, FIBER_IMAGE_NEXT, FID_IMAGES_NEXT, out_dir=OUT_DIR, tile=TILE, itrial=None,
                  magnification=MAGNIFICATION, model=MODEL, nwindow=NWINDOW, threshold=THRESHOLD,
                  niter_recenter=NITER_RECENTER, sat_level=SAT_LEVEL, n_extra=N_EXTRA,
                  min_dist=MIN_DIST, finder_opts=FINDER_OPTS)
tab = lab.offsets_table(res); err = np.hypot(tab.dx_um, tab.dy_um)
print(f"trial {res['itrial']}: fiber - target median {np.nanmedian(err):.1f} um, max {np.nanmax(err):.1f} um")
print('cumulative angles ->', res['json'])
lab.history(OUT_DIR, TILE)



## 4. positioner 별 실제 각도
측정 위치를 회전 중심 기준으로 각도로 되돌리면, 각 positioner 가 명령 각도에서 얼마나 벗어나 있는지(개체별 zero offset)가 나온다.


In [ ]:

from kspec_metrology.analysis.mtlcal import load_configuration
xo, yo, x, y, fid_flag, nfib = load_configuration(res['target_file'], apply_zenith_offset=False)
a1, a2 = cfg.arm_lengths()
X = x[:nfib] + tab.dx_um.values*1e-3; Y = y[:nfib] + tab.dy_um.values*1e-3
alpha, beta = lab.xy2ab(xo[:nfib], yo[:nfib], X, Y, a1, a2)
cmd_a, cmd_b, _ = lab.read_targets(TARGET_FILE)
A = pd.DataFrame(dict(alpha_meas=alpha, beta_meas=beta, dalpha=((alpha - cmd_a + 180) % 360) - 180, dbeta=beta - cmd_b,
                      r_mm=np.hypot(X - xo[:nfib], Y - yo[:nfib])), index=tab.index)
print(f"alpha offset median {A.dalpha.median():.1f} deg (10-90%: {np.nanpercentile(A.dalpha, 10):.1f}..{np.nanpercentile(A.dalpha, 90):.1f}), "
      f"beta offset median {A.dbeta.median():.1f} deg")
A.round(2)



## 5. 단계별 그림
fiber 이미지 그림에서 하늘색 = 매칭된 fiber, 주황 = fiducial 노출에서 잰 fiducial 자리 (이 이미지에서는 쓰지 않음).


In [ ]:

info = lab.show_steps(res)
